# Keyword-Based Disease Annotator

Annotate `data/processed/merged_data.csv` using keyword presence in `cleaned_text`.

**Label order contract:** `["AURI", "PN", "TB", "COVID"]`

This notebook:
- loads base keyword CSVs from `docs/keywords/`
- normalizes keywords and `cleaned_text`
- creates EDA-ready kept and dropped row DataFrames
- creates `annotate`, `AURI`, `PN`, `TB`, and `COVID`
- writes `data/processed/merged_data_annotated.csv`


In [1]:
import ast
import csv
from pathlib import Path

import pandas as pd

LABELS = ["AURI", "PN", "TB", "COVID"]
CWD = Path.cwd()
ROOT_DIR = next((path for path in [CWD, *CWD.parents] if (path / ".git").exists()), CWD)
KEYWORD_FILES = {
    "AURI": ROOT_DIR / "docs/keywords/ri_keywords.csv",
    "PN": ROOT_DIR / "docs/keywords/pn_keywords.csv",
    "TB": ROOT_DIR / "docs/keywords/tb_keywords.csv",
    "COVID": ROOT_DIR / "docs/keywords/covid_keywords.csv",
}
INPUT_PATH = ROOT_DIR / "data/processed/merged_data.csv"
OUTPUT_PATH = ROOT_DIR / "data/processed/merged_data_annotated.csv"


In [2]:
def normalize_text(value):
    if value is None:
        return ""
    text = str(value).strip().lower()
    return "" if text == "nan" else text


def load_keywords(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing keyword file: {path}")

    keywords = []
    seen = set()

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            for cell in row:
                keyword = normalize_text(cell)
                if keyword and keyword not in seen:
                    keywords.append(keyword)
                    seen.add(keyword)

    if not keywords:
        raise ValueError(f"Keyword file is empty after normalization: {path}")

    return keywords


def annotate_text(cleaned_text, keyword_map):
    text = normalize_text(cleaned_text)
    vector = [int(any(keyword in text for keyword in keyword_map[label])) for label in LABELS]
    return vector


In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {INPUT_PATH}")

with INPUT_PATH.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fieldnames = reader.fieldnames or []

if "cleaned_text" not in fieldnames:
    raise KeyError("Expected `cleaned_text` column in the source dataset.")

keyword_map = {label: load_keywords(path) for label, path in KEYWORD_FILES.items()}
keyword_counts = {label: len(keywords) for label, keywords in keyword_map.items()}

print({
    "cwd": str(CWD),
    "root_dir": str(ROOT_DIR),
    "rows": len(rows),
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "keyword_counts": keyword_counts,
})


{'cwd': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/notebooks', 'root_dir': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus', 'rows': 33672, 'input_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/processed/merged_data.csv', 'output_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/processed/merged_data_annotated.csv', 'keyword_counts': {'AURI': 71, 'PN': 80, 'TB': 78, 'COVID': 115}}


In [4]:
annotated_rows = []
dropped_rows = []
for row in rows:
    vector = annotate_text(row.get("cleaned_text", ""), keyword_map)
    enriched = dict(row)
    enriched["annotate"] = str(vector)
    for label, value in zip(LABELS, vector):
        enriched[label] = value

    if any(vector):
        annotated_rows.append(enriched)
    else:
        dropped_rows.append(enriched)

output_fieldnames = fieldnames + ["annotate"] + LABELS
annotated_df = pd.DataFrame(annotated_rows, columns=output_fieldnames)
dropped_df = pd.DataFrame(dropped_rows, columns=output_fieldnames)

print({"kept_rows": len(annotated_df), "dropped_rows": len(dropped_df)})
print("Annotated rows preview")
display(annotated_df.head())
print("Dropped rows preview")
display(dropped_df.head())


{'kept_rows': 23673, 'dropped_rows': 9999}
Annotated rows preview


,id,source,external_post_id,text,cleaned_text,language,date_posted,date_collected,view_count,like_count,share_count,comment_count,created_at,updated_at,annotate,AURI,PN,TB,COVID
0,ddc8b185-59a7-489f-a07b-c9cc316d3cbf,twitter,1969912891831939529,anong gusto mo gawin niya makipagbarda sa mga ...,anong gusto mo gawin niya makipagbarda sa mga ...,tl,2025-09-21 23:53:27+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:53:27+00:00,2026-05-15 06:40:10.855982+00:00,"[1, 1, 1, 1]",1,1,1,1
1,4f3e6fc7-05c1-4676-ad86-1ad5601fe9ab,twitter,1969906492611645849,grabe na hutoy sa ubo thanks mama cels sa pag ...,grabe na hutoy sa ubo thanks mama cels sa pag ...,sh,2025-09-21 23:28:01+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:28:01+00:00,2026-05-15 06:40:10.855982+00:00,"[1, 1, 1, 1]",1,1,1,1
2,37416922-9e30-4f5c-9e71-f62d39463185,twitter,1969901404925087862,"may caption na lahat yung video, hindi pa rin ...","may caption na lahat yung video, hindi pa rin ...",tl,2025-09-21 23:07:48+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:07:48+00:00,2026-05-15 06:40:10.855982+00:00,"[1, 1, 1, 1]",1,1,1,1
3,4916ae5c-db59-4687-a8e0-993b34561035,twitter,1969900001779495319,@NellyGBasco @lorasantos11 REST MUNA... NEED R...,rest muna... need rest.. inihit ulit ng ubo......,en,2025-09-21 23:02:14+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:02:14+00:00,2026-05-15 06:40:10.855982+00:00,"[1, 1, 1, 1]",1,1,1,1
4,ba5f7927-5b89-4ae4-83ed-308acbab7ea3,twitter,1969899015228760418,"In the UAE, UBO filings require entities to id...","in the uae, ubo filings require entities to id...",en,2025-09-21 22:58:18+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 22:58:18+00:00,2026-05-15 06:40:10.855982+00:00,"[1, 1, 1, 1]",1,1,1,1


Dropped rows preview


,id,source,external_post_id,text,cleaned_text,language,date_posted,date_collected,view_count,like_count,share_count,comment_count,created_at,updated_at,annotate,AURI,PN,TB,COVID
0,960f7e1c-ae2e-4fb4-b1b0-03c7c4b9f8dd,twitter,1969914468458185043,https://t.co/jbO5TDH1nM 牛乳石鹸コラボユニボールワンP 1家族1...,p 11,ja,2025-09-21 23:59:43+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:59:43+00:00,2026-05-15 06:40:10.855982+00:00,"[0, 0, 0, 0]",0,0,0,0
1,f134e60b-9074-4dcc-a7b8-17ffe3bbd2b9,twitter,1969906030298763648,@ubo_ub @MaseDenver Trautman sucks ass,trautman sucks ass,es,2025-09-21 23:26:11+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:26:11+00:00,2026-05-15 06:40:10.855982+00:00,"[0, 0, 0, 0]",0,0,0,0
2,ba7183c4-041f-40b7-b9b7-4298d86c33f5,twitter,1969904337368256777,@MaseDenver @MaseDenver thoughts on the offici...,thoughts on the officiating crew tonight? that...,en,2025-09-21 23:19:27+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:19:27+00:00,2026-05-15 06:40:10.855982+00:00,"[0, 0, 0, 0]",0,0,0,0
3,bc07e5b3-905c-4da0-84a4-2fef0b6bb805,twitter,1969903982794743937,@VicLombardi @Broncos @AltitudeTV Horrible gam...,horrible game when it comes to the vibe and a ...,en,2025-09-21 23:18:03+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:18:03+00:00,2026-05-15 06:40:10.855982+00:00,"[0, 0, 0, 0]",0,0,0,0
4,7aefa1c1-7c3c-4792-94a1-a552e341dde5,twitter,1969902984105787481,@ZacStevensDNVR Shit game. There’s no other wa...,shit game. theres no other way to put it. refs...,en,2025-09-21 23:14:05+00:00,2026-05-15 06:40:10.855982+00:00,,,,,2025-09-21 23:14:05+00:00,2026-05-15 06:40:10.855982+00:00,"[0, 0, 0, 0]",0,0,0,0


In [5]:
parsed_vectors = annotated_df["annotate"].apply(ast.literal_eval).tolist()
dropped_vectors = dropped_df["annotate"].apply(ast.literal_eval).tolist()
vector_lengths = [len(values) for values in parsed_vectors]
binary_ok = [all(item in (0, 1) for item in values) for values in parsed_vectors]

assert len(annotated_df) + len(dropped_df) == len(rows)
assert all(length == len(LABELS) for length in vector_lengths)
assert all(binary_ok)
assert (annotated_df[LABELS].astype(int).values == pd.DataFrame(parsed_vectors, columns=LABELS).values).all()
assert all(any(value == 1 for value in vector) for vector in parsed_vectors)
assert all(not any(vector) for vector in dropped_vectors)

print("Annotation contract validated.")
print({"input_rows": len(rows), "output_rows": len(annotated_df), "dropped_rows": len(dropped_df)})
print(annotated_df[LABELS].astype(int).sum().to_dict())


Annotation contract validated.
{'input_rows': 33672, 'output_rows': 23673, 'dropped_rows': 9999}
{'AURI': 21460, 'PN': 7181, 'TB': 8860, 'COVID': 10879}


In [6]:
annotated_df.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(annotated_df):,} annotated rows to {OUTPUT_PATH}")


Exported 23,673 annotated rows to /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/processed/merged_data_annotated.csv


In [7]:
# Spot checks for single-label, multi-label, and dropped rows.
eda_columns = ["cleaned_text", "annotate"] + LABELS
label_sums = annotated_df[LABELS].astype(int).sum(axis=1)

single_label_df = annotated_df.loc[label_sums == 1, eda_columns].head(3)
multi_label_df = annotated_df.loc[label_sums > 1, eda_columns].head(3)
dropped_sample_df = dropped_df.loc[:, eda_columns].head(3)

print("Single-label examples")
display(single_label_df)
print("Multi-label examples")
display(multi_label_df)
print("Dropped rows examples")
display(dropped_sample_df)


Single-label examples


,cleaned_text,annotate,AURI,PN,TB,COVID
26,ona jurila kolegu nozem da ubada i jos nije u ...,"[1, 0, 0, 0]",1,0,0,0
103,izofrena cepanja i agresivne ispade imala je i...,"[1, 0, 0, 0]",1,0,0,0
318,hate na hate ko makarinig ng ai talaga. pero t...,"[1, 0, 0, 0]",1,0,0,0


Multi-label examples


,cleaned_text,annotate,AURI,PN,TB,COVID
0,anong gusto mo gawin niya makipagbarda sa mga ...,"[1, 1, 1, 1]",1,1,1,1
1,grabe na hutoy sa ubo thanks mama cels sa pag ...,"[1, 1, 1, 1]",1,1,1,1
2,"may caption na lahat yung video, hindi pa rin ...","[1, 1, 1, 1]",1,1,1,1


Dropped rows examples


,cleaned_text,annotate,AURI,PN,TB,COVID
0,p 11,"[0, 0, 0, 0]",0,0,0,0
1,trautman sucks ass,"[0, 0, 0, 0]",0,0,0,0
2,thoughts on the officiating crew tonight? that...,"[0, 0, 0, 0]",0,0,0,0
